In [126]:
import pandas as pd

In [127]:
# Webscrape links into dataframes
player_stats = pd.read_html('https://www.basketball-reference.com/leagues/NBA_2024_advanced.html', header=0, attrs={'id':'advanced'})[0]
player_stats2 = pd.read_html('https://www.basketball-reference.com/leagues/NBA_2024_totals.html', header=0, attrs={'id':'totals_stats'})[0]
east_stats = pd.read_html('https://www.basketball-reference.com/leagues/NBA_2024.html', header=0, attrs={'id':'confs_standings_E'})[0]
west_stats = pd.read_html('https://www.basketball-reference.com/leagues/NBA_2024.html', header=0, attrs={'id':'confs_standings_W'})[0]
mvp_stats = pd.read_html('https://www.basketball-reference.com/awards/awards_2024.html', header=1, attrs={'id':'mvp'})[0]

In [128]:
# Variables
nba_teams = {
    "Atlanta Hawks": "ATL",
    "Boston Celtics": "BOS",
    "Brooklyn Nets": "BRK",  # Sometimes "BKN"
    "Chicago Bulls": "CHI",
    "Charlotte Hornets": "CHO",  # Sometimes "CHA"
    "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL",
    "Denver Nuggets": "DEN",
    "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW",
    "Houston Rockets": "HOU",
    "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC",
    "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA",
    "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP",
    "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL",
    "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHO",
    "Portland Trail Blazers": "POR",
    "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR",
    "Utah Jazz": "UTA",
    "Washington Wizards": "WAS"
}

In [169]:
# Feature Selection/Engineering
east_stats_ = east_stats.rename(columns={'Eastern Conference':'Team'}).loc[:,['Team', 'W/L%']]
east_stats_['Team'] = east_stats_['Team'].str.replace('*','', regex=True)
east_stats_['Seed'] = east_stats.index + 1
west_stats_ = west_stats.rename(columns={'Western Conference':'Team'}).loc[:,['Team', 'W/L%']]
west_stats_['Team'] = west_stats_['Team'].str.replace('*','', regex=True)
west_stats_['Seed'] = west_stats.index + 1

team_stats_ = pd.concat([east_stats_, west_stats_])
team_stats_.replace(nba_teams, inplace=True)

player_stats_ = player_stats.loc[:,['Player', 'Team', 'G', 'GS', 'MP', 'PER', 'TS%',
       '3PAr', 'FTr', 'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%','USG%', 'OWS', 'DWS', 'WS', 'WS/48', 'OBPM', 'DBPM', 'BPM', 'VORP']].iloc[:-1]
player_stats_['MP'] = round(player_stats_['MP'] / player_stats_['G'], 1)
player_stats_.drop_duplicates('Player', keep='first', inplace=True)

player_stats_teams = player_stats.copy()
player_stats_teams = player_stats_teams.loc[:, ['Player', 'Team']]
player_stats_teams.drop_duplicates('Player', keep='last', inplace=True)

player_stats_2 = player_stats2.loc[:, ['Player','FGA', 'FG%', '3P%', 'FT%','eFG%', 'PTS']].iloc[:-1]
player_stats_2.drop_duplicates('Player', keep='first', inplace=True)
player_stats_ = player_stats_.merge(player_stats_teams, on='Player', suffixes=('', '_Correct'))
player_stats_['Team'] = player_stats_['Team_Correct']
player_stats_.drop(columns=['Team_Correct'], inplace=True)

mvp_stats_ = mvp_stats.loc[:,['Player', 'Tm', 'Share', 'MP', 'FG%', '3P%', 'FT%', 'WS', 'WS/48']]
mvp_stats_.rename(columns={'Tm':'Team'}, inplace=True)
#mvp_stats_['Tm'].replace(nba_teams, inplace=True)
mvp_stats_

,Player,Team,Share,MP,FG%,3P%,FT%,WS,WS/48
0,Nikola Jokić,DEN,0.935,34.6,0.583,0.359,0.817,17.0,0.299
1,Shai Gilgeous-Alexander,OKC,0.646,34.0,0.535,0.353,0.874,14.6,0.275
2,Luka Dončić,DAL,0.572,37.5,0.487,0.382,0.786,12.0,0.220
3,Giannis Antetokounmpo,MIL,0.194,35.2,0.611,0.274,0.657,13.2,0.246
4,Jalen Brunson,NYK,0.143,35.4,0.479,0.401,0.847,11.2,0.198
5,Jayson Tatum,BOS,0.087,35.7,0.471,0.376,0.833,10.4,0.189
6,Anthony Edwards,MIN,0.018,35.1,0.461,0.357,0.836,7.5,0.130
7,Domantas Sabonis,SAC,0.003,35.7,0.594,0.379,0.704,12.6,0.206
8,Kevin Durant,PHO,0.001,37.2,0.523,0.413,0.856,8.3,0.142


In [170]:
display(team_stats_)
display(player_stats_)
display(player_stats_2)

,Team,W/L%,Seed
0,BOS,0.780,1
1,NYK,0.610,2
2,MIL,0.598,3
3,CLE,0.585,4
4,ORL,0.573,5
5,IND,0.573,6
6,PHI,0.573,7
7,MIA,0.561,8
8,CHI,0.476,9
9,ATL,0.439,10


,Player,Team,G,GS,MP,PER,TS%,3PAr,FTr,ORB%,DRB%,TRB%,AST%,STL%,BLK%,TOV%,USG%,OWS,DWS,WS,WS/48,OBPM,DBPM,BPM,VORP
0,DeMar DeRozan,CHI,79.0,79.0,37.8,19.7,0.584,0.166,0.452,1.6,11.3,6.4,21.8,1.5,1.5,7.7,25.8,7.0,2.2,9.2,0.147,2.1,-0.3,1.8,2.8
1,Domantas Sabonis,SAC,82.0,82.0,35.7,23.2,0.637,0.081,0.389,11.0,32.3,21.4,33.9,1.2,1.5,17.9,22.2,8.6,4.0,12.6,0.206,4.0,2.4,6.5,6.2
2,Coby White,CHI,79.0,78.0,36.5,14.5,0.570,0.460,0.215,1.7,12.4,6.9,20.8,0.9,0.6,11.1,22.7,3.1,1.6,4.7,0.078,0.7,-1.3,-0.7,0.9
3,Mikal Bridges,BRK,82.0,82.0,34.8,14.9,0.560,0.457,0.245,2.5,12.0,7.1,16.3,1.4,0.9,10.3,24.3,2.1,2.1,4.2,0.070,0.7,-1.0,-0.4,1.2
4,Paolo Banchero,ORL,80.0,80.0,35.0,17.3,0.546,0.249,0.397,3.4,20.0,11.6,25.2,1.3,1.6,13.0,29.7,1.3,4.0,5.3,0.090,1.3,0.0,1.3,2.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
567,Malcolm Cazalon,DET,1.0,0.0,3.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,-0.012,-6.0,-3.2,-9.1,0.0
568,Jalen Crutcher,NOP,1.0,0.0,3.0,-12.6,0.000,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.5,0.0,0.0,0.0,-0.334,-18.5,-7.8,-26.2,0.0
569,Dmytro Skapintsev,NYK,2.0,0.0,1.0,-19.3,0.000,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.6,0.0,0.0,0.0,-0.483,-16.0,-9.8,-25.9,0.0
570,Justin Jackson,MIN,2.0,0.0,0.5,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.031,-6.3,-1.2,-7.5,0.0


,Player,FGA,FG%,3P%,FT%,eFG%,PTS
0,Luka Dončić,1652.0,0.487,0.382,0.786,0.573,2370.0
1,Shai Gilgeous-Alexander,1487.0,0.535,0.353,0.874,0.567,2254.0
2,Giannis Antetokounmpo,1369.0,0.611,0.274,0.657,0.624,2222.0
3,Jalen Brunson,1648.0,0.479,0.401,0.847,0.543,2212.0
4,Nikola Jokić,1411.0,0.583,0.359,0.817,0.612,2085.0
...,...,...,...,...,...,...,...
730,Danny Green,2.0,0.000,0.000,NaN,0.000,0.0
731,Ron Harper Jr.,0.0,NaN,NaN,NaN,NaN,0.0
732,Justin Jackson,0.0,NaN,NaN,NaN,NaN,0.0
733,Dmytro Skapintsev,1.0,0.000,NaN,NaN,0.000,0.0


In [167]:
pd.set_option('display.max_columns', None)
a = pd.merge(player_stats_, player_stats_2, how='outer')
b = pd.merge(a, mvp_stats_, how='outer')
c = pd.merge(b, team_stats_, how='outer')
c

,Player,Team,G,GS,MP,PER,TS%,3PAr,FTr,ORB%,DRB%,TRB%,AST%,STL%,BLK%,TOV%,USG%,OWS,DWS,WS,WS/48,OBPM,DBPM,BPM,VORP,FGA,FG%,3P%,FT%,eFG%,PTS,Share,W/L%,Seed
0,DeMar DeRozan,CHI,79.0,79.0,37.8,19.7,0.584,0.166,0.452,1.6,11.3,6.4,21.8,1.5,1.5,7.7,25.8,7.0,2.2,9.2,0.147,2.1,-0.3,1.8,2.8,1355.0,0.480,0.333,0.853,0.507,1897.0,0.000,0.476,9.0
1,Coby White,CHI,79.0,78.0,36.5,14.5,0.570,0.460,0.215,1.7,12.4,6.9,20.8,0.9,0.6,11.1,22.7,3.1,1.6,4.7,0.078,0.7,-1.3,-0.7,0.9,1209.0,0.447,0.376,0.838,0.534,1509.0,0.000,0.476,9.0
2,Nikola Vučević,CHI,76.0,74.0,34.3,17.7,0.540,0.258,0.107,8.8,25.8,17.1,15.2,1.0,2.4,8.6,23.3,2.7,2.8,5.4,0.100,0.8,-0.7,0.1,1.4,1211.0,0.484,0.294,0.822,0.522,1370.0,0.000,0.476,9.0
3,Ayo Dosunmu,CHI,76.0,37.0,29.1,13.4,0.604,0.410,0.146,2.7,8.3,5.5,15.6,1.5,1.7,11.9,17.2,2.6,1.5,4.1,0.089,-0.6,-0.2,-0.8,0.7,719.0,0.501,0.403,0.810,0.583,924.0,0.000,0.476,9.0
4,Alex Caruso,CHI,71.0,57.0,28.7,14.5,0.613,0.616,0.177,3.3,11.8,7.5,16.4,2.9,3.5,14.9,14.7,2.3,2.6,4.9,0.115,0.2,2.3,2.5,2.3,541.0,0.468,0.408,0.760,0.593,715.0,0.000,0.476,9.0
5,Andre Drummond,CHI,79.0,10.0,17.1,23.0,0.576,0.006,0.454,21.5,37.4,29.3,4.1,2.7,3.6,12.3,21.3,2.3,2.6,4.9,0.175,0.8,-0.7,0.1,0.7,480.0,0.556,0.000,0.592,0.556,663.0,0.000,0.476,9.0
6,Patrick Williams,CHI,43.0,30.0,27.3,11.0,0.553,0.412,0.184,4.3,11.7,7.9,7.7,1.7,2.9,12.8,16.6,0.1,1.0,1.2,0.048,-2.1,-0.1,-2.3,-0.1,359.0,0.443,0.399,0.788,0.525,429.0,0.000,0.476,9.0
7,Torrey Craig,CHI,53.0,14.0,19.8,10.8,0.568,0.610,0.143,7.2,16.1,11.6,7.3,1.4,2.1,10.7,12.5,1.0,0.9,1.9,0.087,-1.8,-0.2,-2.0,0.0,251.0,0.430,0.392,0.750,0.550,303.0,0.000,0.476,9.0
8,Jevon Carter,CHI,72.0,0.0,13.9,8.5,0.485,0.641,0.019,1.2,5.6,3.3,12.9,1.8,1.6,8.9,17.7,-0.5,0.7,0.2,0.009,-3.3,-0.7,-4.0,-0.5,365.0,0.378,0.329,0.571,0.484,357.0,0.000,0.476,9.0
9,Zach LaVine,CHI,25.0,23.0,34.9,15.1,0.578,0.449,0.274,1.0,15.8,8.3,16.8,1.2,0.9,11.0,23.8,0.8,0.6,1.5,0.080,0.8,-0.7,0.1,0.5,376.0,0.452,0.349,0.854,0.531,487.0,0.000,0.476,9.0
